# Joint Effect Supplement

Run cells **1 → 2 → 3 → 3b → 4** in order.

**After GPU disconnect:** run **Cell 4b** first (downloads checkpoint), then Cell 4 to resume.

**Cell 1 must show `Package version: v4 OK`**

In [ ]:
# CELL 1: Mount Drive + verify package version (MUST be v4)
from google.colab import drive
drive.mount('/content/drive')

import os, re
FOLDER = '/content/drive/MyDrive/joint_supplement_colab'
os.chdir(FOLDER)
print('Working directory:', os.getcwd())

need = ['run_generation_supplement.py', 'diagnose_checkpoint.py', 'clean_checkpoint.py',
        'backup_checkpoint.py', 'test_setup.py', 'prompts_gender.csv', 'data/v1_gender_existing.csv']
for f in need:
    print(f'  {f}:', 'OK' if os.path.exists(f) else 'MISSING — re-upload entire folder!')

src = open('run_generation_supplement.py').read()
m = re.search(r'PACKAGE_VERSION = "(\d+)"', src)
ver = m.group(1) if m else '?'
ok = ver == '4'
print(f'  Package version: v{ver}', 'OK' if ok else 'STOP — upload v4 folder!')
assert ok, 'Wrong package on Drive. Upload joint_supplement_colab from your computer.'

ckpt = 'outputs/generations_supplement.csv'
if os.path.exists(ckpt):
    print(f'  {ckpt}: {os.path.getsize(ckpt)/1e6:.2f} MB')
else:
    print(f'  {ckpt}: missing yet')

In [ ]:
# CELL 2: Install + HuggingFace login (input box — do NOT paste token in code)
!pip install -q "transformers>=4.40" "accelerate>=0.28" "bitsandbytes>=0.43" huggingface_hub scipy statsmodels matplotlib pandas

from getpass import getpass
from huggingface_hub import login, get_token

token = getpass("Paste HuggingFace token (Write access), then press Enter: ")
login(token=token.strip(), add_to_git_credential=False)

assert get_token(), "Login failed — create a Write token at https://huggingface.co/settings/tokens"
print("HuggingFace login OK")
print("Also accept Llama license: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct")

In [ ]:
# CELL 3: Test GPU + HuggingFace + Llama access (MUST PASS before Cell 4)
!python test_setup.py

In [ ]:
# CELL 3b: Checkpoint diagnostic (run before Cell 4)
!python diagnose_checkpoint.py
!python clean_checkpoint.py

In [ ]:
# CELL 4: Generate missing stories (~970). Re-run this cell if disconnected.
!python run_generation_supplement.py

In [ ]:
# CELL 4b: BACKUP — run after Cell 4 stops (CPU is fine, no GPU needed)
!python backup_checkpoint.py

from google.colab import files
files.download("outputs/generations_supplement.csv")
print("If download started, check your browser Downloads folder.")

In [ ]:
# CELL 5: Merge + analyze
!python merge_and_analyze.py

In [ ]:
# CELL 6: Show figures
from IPython.display import Image, display
from pathlib import Path
for fig in sorted(Path('outputs/figures').glob('*.png')):
    print(fig.name)
    display(Image(str(fig)))

In [ ]:
# CELL 7: Download zip
import shutil
from google.colab import files
shutil.make_archive('joint_supplement_results', 'zip', 'outputs')
files.download('joint_supplement_results.zip')